In [1]:
%load_ext cudf.pandas
import numpy as np
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
import pandas as pd
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook

Enabled rmm statistics


In [3]:
passmark = 40
df = pd.read_csv("/home/dias-benchmarks/notebooks/spscientist/student-performance-in-exams/input/StudentsPerformance.csv")
df = pd.concat([df]*1000)
df.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 1000000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count    Dtype
---  ------                       --------------    -----
 0   gender                       1000000 non-null  object
 1   race/ethnicity               1000000 non-null  object
 2   parental level of education  1000000 non-null  object
 3   lunch                        1000000 non-null  object
 4   test preparation course      1000000 non-null  object
 5   math score                   1000000 non-null  object
 6   reading score                1000000 non-null  object
 7   writing score                1000000 non-null  object
dtypes: object(8)
memory usage: 83.8+ MB


In [4]:
factor = 100
df = pd.concat([df]*factor)
df.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 100000000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Dtype
---  ------                       -----
 0   gender                       object
 1   race/ethnicity               object
 2   parental level of education  object
 3   lunch                        object
 4   test preparation course      object
 5   math score                   object
 6   reading score                object
 7   writing score                object
dtypes: object(8)
memory usage: 8.2+ GB


In [5]:
%%ProfileMem
### cell 0 ###
_ = df.isnull().sum()

Cell runtime: 24.776697158813477
Memory usage before cell: 8.19 GB (8382.89 MB)
Memory usage after cell: 8.93 GB (9145.83 MB)
Peak usage of cell: 8.93 GB (9145.86 MB)


In [6]:
%%ProfileMem
### cell 1 ###

df['math score'] = df['math score'].astype(float).fillna(-1)  # Replace NaNs with a default value
# Vectorized conditional assignment using cuDF
df['Math_PassStatus'] = (df['math score'] >= passmark).astype("str").replace({'True': 'P', 'False': 'F'})

# Use cuDF's optimized value_counts()
_ = df['Math_PassStatus'].value_counts()

Cell runtime: 257.429838180542
Memory usage before cell: 8.93 GB (9145.83 MB)
Memory usage after cell: 8.84 GB (9049.99 MB)
Peak usage of cell: 13.33 GB (13646.34 MB)


In [7]:
### cell 2 ###
df['reading score'] = df['reading score'].astype(float).fillna(-1)  # Replace NaNs if needed
# Use cuDF's vectorized conditional assignment
df['Reading_PassStatus'] = (df['reading score'] >= passmark).astype("str").replace({'True': 'P', 'False': 'F'})
# Use cuDF's value_counts() (faster than pandas on GPU)
_ = df['Reading_PassStatus'].value_counts()


In [8]:
%%ProfileMem
### cell 3 ###
df['writing score'] = df['writing score'].astype('float32')
# Use cuDF's boolean indexing + direct assignment (avoids unnecessary operations)
df['Writing_PassStatus'] = 'F'  # Default all to 'F' first
df.loc[df['writing score'] >= passmark, 'Writing_PassStatus'] = 'P'  # Assign 'P' where needed
# Efficient value_counts() with cuDF
_ = df['Writing_PassStatus'].value_counts()

Cell runtime: 200.5321979522705
Memory usage before cell: 9.49 GB (9715.94 MB)
Memory usage after cell: 9.77 GB (10000.71 MB)
Peak usage of cell: 14.06 GB (14402.51 MB)


In [9]:
%%ProfileMem
### cell 4 ###
df['OverAll_PassStatus'] = ((df['Math_PassStatus'] == 'P') & 
                            (df['Reading_PassStatus'] == 'P') & 
                            (df['Writing_PassStatus'] == 'P')).astype("str").replace({'True': 'P', 'False': 'F'})

# Optimized cuDF value_counts()
_ = df['OverAll_PassStatus'].value_counts()

Cell runtime: 138.5209560394287
Memory usage before cell: 9.77 GB (10000.71 MB)
Memory usage after cell: 10.23 GB (10477.54 MB)
Peak usage of cell: 14.72 GB (15074.95 MB)


In [10]:
%%ProfileMem
### cell 5 ###
df['Total_Marks'] = df['math score']+df['reading score']+df['writing score']
df['Percentage'] = df['Total_Marks']/3

Cell runtime: 62.285423278808594
Memory usage before cell: 10.23 GB (10477.54 MB)
Memory usage after cell: 11.72 GB (12003.42 MB)
Peak usage of cell: 11.72 GB (12003.42 MB)


In [ ]:
%%ProfileMem
### cell 6 ###
df['Grade'] = 'F'

# Apply vectorized conditional assignments
df.loc[df['OverAll_PassStatus'] == 'F', 'Grade'] = 'F'
df.loc[(df['Percentage'] >= 80) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'A'
df.loc[(df['Percentage'] >= 70) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'B'
df.loc[(df['Percentage'] >= 60) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'C'
df.loc[(df['Percentage'] >= 50) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'D'
df.loc[(df['Percentage'] >= 40) & (df['OverAll_PassStatus'] != 'F'), 'Grade'] = 'E'

# Efficient value counts
_ = df['Grade'].value_counts()